# FusionModel Training on PTB‑XL (ECG + Metadata)

## Sections
1. Load PTB‑XL ECG + Metadata
2. Build Fusion Dataloaders
3. Load Pretrained ECG & Metadata Models
4. Build Fusion Model
5. Train Fusion Classifier
6. Training of the fusion model
7. Evaluation of the fusion model
8. Full training
9. Save best model

## 0. Setup

In [16]:
import sys
sys.path.append("C:/Users/inaki/Desktop/TFG/Code/ptbxl_project")

print(sys.executable)


c:\Users\inaki\anaconda3\envs\ptbxl\python.exe


## 1. Load PTB‑XL ECG + Metadata

In [17]:
import pandas as pd
from utils import load_ptbxl
from utils.data_loader import load_metadata

# Path and sampling rate
PATH = "C:/Users/inaki/Desktop/TFG/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"
SAMPLING_RATE = 100

# -------------------------------
# Load ECG signals + labels
# -------------------------------
X_train, X_val, X_test, y_train, y_val, y_test, classes = load_ptbxl(
    PATH, SAMPLING_RATE
)

print("ECG shapes:", X_train.shape, X_val.shape, X_test.shape)

# -------------------------------
# Load metadata CSV
# -------------------------------
meta_full = pd.read_csv(PATH + "ptbxl_database.csv")
meta_full["recording_date"] = pd.to_datetime(meta_full["recording_date"])

# -------------------------------
# Official PTB‑XL fold splits
# -------------------------------
train_folds = list(range(1, 9))   # folds 1–8
val_fold = 9                      # fold 9
test_fold = 10                    # fold 10

# -------------------------------
# Split metadata using folds
# -------------------------------
meta_train = load_metadata(meta_full[meta_full.strat_fold.isin(train_folds)])
meta_val   = load_metadata(meta_full[meta_full.strat_fold == val_fold])
meta_test  = load_metadata(meta_full[meta_full.strat_fold == test_fold])

print("Metadata shapes:", meta_train.shape, meta_val.shape, meta_test.shape)


ECG shapes: (17418, 1000, 12) (2183, 1000, 12) (2198, 1000, 12)
Metadata shapes: (17418, 10) (2183, 10) (2198, 10)


## 2. Build Fusion Dataloaders

In [18]:
from torch.utils.data import DataLoader
from utils.data_loader import FusionDataset

# Reset metadata indices to align with ECG arrays
meta_train_fixed = meta_train.reset_index(drop=True)
meta_val_fixed   = meta_val.reset_index(drop=True)
meta_test_fixed  = meta_test.reset_index(drop=True)

# Build fusion datasets
fusion_train_ds = FusionDataset(X_train, meta_train_fixed, y_train)
fusion_val_ds   = FusionDataset(X_val, meta_val_fixed, y_val)
fusion_test_ds  = FusionDataset(X_test, meta_test_fixed, y_test)

# Build dataloaders
fusion_train_dl = DataLoader(fusion_train_ds, batch_size=32, shuffle=True)
fusion_val_dl   = DataLoader(fusion_val_ds, batch_size=32)
fusion_test_dl  = DataLoader(fusion_test_ds, batch_size=32)

## 3. Load Pretrained ECG & Metadata Models

In [19]:
import torch
from models.meta_mlp import MetaMLP
from models.xresnet1d import xresnet1d101

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load ECG model WITH full head intact (outputs 5-dim logits, no Identity replacement)
ecg_model = xresnet1d101(input_channels=12, num_classes=5)
ecg_model.load_state_dict(torch.load("../outputs/best_ecg_model.pt", map_location=DEVICE))
ecg_model = ecg_model.to(DEVICE)

# Verify — should print (1, 5)
with torch.no_grad():
    sample = torch.randn(1, 12, 1000, device=DEVICE)
    feat = ecg_model(sample)
    print("ecg_feat shape:", feat.shape)  # expected: (1, 5)

meta_model = MetaMLP(in_features=meta_train.shape[1]).to(DEVICE)

C:\Users\inaki\AppData\Local\Temp\ipykernel_6540\2848032710.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ecg_model.load_state_dict(torch.load("../outputs/best_ecg_mod

ecg_feat shape: torch.Size([1, 5])


## 4. Build Fusion Model

In [20]:
from models.fusion import FusionModel

num_classes = len(classes)   # 5

fusion_model = FusionModel(
    ecg_model  = ecg_model,
    meta_model = meta_model,
    num_classes= num_classes
).to(DEVICE)

# Verify fusion dim — should print 37
print("Fusion dim:", fusion_model.classifier[0].weight.shape[1])  # 37
print(f"Total parameters: {sum(p.numel() for p in fusion_model.parameters())}")

Fusion dim: 37
Total parameters: 1811498


## 5. Train Fusion Classifier

In [21]:
import importlib
import models.fusion as fusion_module
importlib.reload(fusion_module)

from models.fusion import FusionModel

# RECREATE MODEL
fusion_model = FusionModel(
    ecg_model=ecg_model,
    meta_model=meta_model,
    num_classes=num_classes
).to(DEVICE)

import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

# Freeze ECG model (recommended)
for param in fusion_model.ecg.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-4)

EPOCHS = 20
PATIENCE = 5   # stop if no improvement for 5 epochs

best_val_loss = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    # ---------------------
    # TRAIN
    # ---------------------
    fusion_model.train()
    total_loss = 0

    for ecg, meta, y in fusion_train_dl:
        ecg, meta, y = ecg.to(DEVICE), meta.to(DEVICE), y.to(DEVICE)
        y = y.float()

        optimizer.zero_grad()
        preds = fusion_model(ecg, meta)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(fusion_train_dl)

    # ---------------------
    # VALIDATION
    # ---------------------
    fusion_model.eval()
    val_loss = 0

    with torch.no_grad():
        for ecg, meta, y in fusion_val_dl:
            ecg, meta, y = ecg.to(DEVICE), meta.to(DEVICE), y.to(DEVICE)
            y = y.float()

            preds = fusion_model(ecg, meta)
            loss = criterion(preds, y)
            val_loss += loss.item()

    val_loss = val_loss / len(fusion_val_dl)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # ---------------------
    # EARLY STOPPING LOGIC
    # ---------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0

        # Save best model
        torch.save(fusion_model.state_dict(), "../outputs/best_fusion_model.pt")
        print("✅ Best model saved")

    else:
        epochs_no_improve += 1
        print(f"⚠️ No improvement ({epochs_no_improve}/{PATIENCE})")

        if epochs_no_improve >= PATIENCE:
            print("🛑 Early stopping triggered")
            break

Epoch 1/20 | Train Loss: 2.1545 | Val Loss: 0.4857
✅ Best model saved
Epoch 2/20 | Train Loss: 0.4789 | Val Loss: 0.4323
✅ Best model saved
Epoch 3/20 | Train Loss: 0.4279 | Val Loss: 0.3880
✅ Best model saved
Epoch 4/20 | Train Loss: 0.3899 | Val Loss: 0.3659
✅ Best model saved
Epoch 5/20 | Train Loss: 0.3672 | Val Loss: 0.3442
✅ Best model saved
Epoch 6/20 | Train Loss: 0.3477 | Val Loss: 0.3411
✅ Best model saved
Epoch 7/20 | Train Loss: 0.3312 | Val Loss: 0.3375
✅ Best model saved
Epoch 8/20 | Train Loss: 0.3190 | Val Loss: 0.3257
✅ Best model saved
Epoch 9/20 | Train Loss: 0.3083 | Val Loss: 0.3191
✅ Best model saved
Epoch 10/20 | Train Loss: 0.3022 | Val Loss: 0.3115
✅ Best model saved
Epoch 11/20 | Train Loss: 0.2977 | Val Loss: 0.3105
✅ Best model saved
Epoch 12/20 | Train Loss: 0.2912 | Val Loss: 0.3012
✅ Best model saved
Epoch 13/20 | Train Loss: 0.2874 | Val Loss: 0.3012
✅ Best model saved
Epoch 14/20 | Train Loss: 0.2851 | Val Loss: 0.2952
✅ Best model saved
Epoch 15/20 | T

## 6. Training of the fusion model

In [22]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

criterion = nn.BCEWithLogitsLoss()  # multi-label
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)

def train_one_epoch(model, dataloader):
    model.train()
    total_loss = 0

    for ecg, meta, labels in dataloader:
        ecg = ecg.to(DEVICE)
        meta = meta.to(DEVICE)
        labels = labels.to(DEVICE).float()

        optimizer.zero_grad()

        outputs = model(ecg, meta)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

## 7. Evaluation of the fusion model

In [23]:
def evaluate(model, dataloader):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for ecg, meta, labels in dataloader:
            ecg = ecg.to(DEVICE)
            meta = meta.to(DEVICE)

            outputs = model(ecg, meta)
            probs = torch.sigmoid(outputs)

            all_probs.append(probs.cpu())
            all_labels.append(labels)

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    preds = (all_probs > 0.5).astype(int)

    # Metrics
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, average="macro")

    try:
        auc = roc_auc_score(all_labels, all_probs, average="macro")
    except:
        auc = float("nan")

    return acc, f1, auc

## 8. Full training

In [24]:
# Step 1 --> Load PTB-XL

from utils.data_loader import load_ptbxl

X_train, X_val, X_test, y_train, y_val, y_test, classes = load_ptbxl(PATH)

# Step 2 --> Load metadata

import pandas as pd
from utils.data_loader import load_metadata

Y = pd.read_csv(PATH + "ptbxl_database.csv", index_col="ecg_id")

meta = load_metadata(Y)

# Split metadata SAME way as ECG
train_folds = list(range(1, 9))
val_fold = 9
test_fold = 10

meta_train = meta[Y.strat_fold.isin(train_folds)]
meta_val = meta[Y.strat_fold == val_fold]
meta_test = meta[Y.strat_fold == test_fold]

# Step 3 --> Create Fusion DataLoaders

from torch.utils.data import DataLoader
from utils.data_loader import FusionDataset

train_dataset = FusionDataset(X_train, meta_train, y_train)
val_dataset = FusionDataset(X_val, meta_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

#
ecg, meta, labels = next(iter(train_loader))

print("ECG:", ecg.shape)
print("Meta:", meta.shape)
print("Labels:", labels.shape)
#

# Step 4 --> Training loop with Early Stopping (based on AUC)

num_epochs = 30
PATIENCE = 5

best_val_auc = 0
epochs_no_improve = 0

for epoch in range(num_epochs):
    train_loss = train_one_epoch(fusion_model, train_loader)

    val_acc, val_f1, val_auc = evaluate(fusion_model, val_loader)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print(f"Val F1: {val_f1:.4f}")
    print(f"Val AUC: {val_auc:.4f}")

    # -------------------------
    # EARLY STOPPING (AUC)
    # -------------------------
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_no_improve = 0

        # Save best model
        torch.save(fusion_model.state_dict(), "../outputs/best_fusion_model.pt")
        print("✅ Best model saved (AUC improved)")

    else:
        epochs_no_improve += 1
        print(f"⚠️ No AUC improvement ({epochs_no_improve}/{PATIENCE})")

        if epochs_no_improve >= PATIENCE:
            print("🛑 Early stopping triggered")
            break

    print("-" * 40)

ECG: torch.Size([32, 12, 1000])
Meta: torch.Size([32, 10])
Labels: torch.Size([32, 5])
Epoch 1/30
Train Loss: 0.3349
Val Accuracy: 0.5598
Val F1: 0.6700
Val AUC: 0.9061
✅ Best model saved (AUC improved)
----------------------------------------
Epoch 2/30
Train Loss: 0.2920
Val Accuracy: 0.5964
Val F1: 0.6782
Val AUC: 0.9106
✅ Best model saved (AUC improved)
----------------------------------------
Epoch 3/30
Train Loss: 0.2535
Val Accuracy: 0.6207
Val F1: 0.7079
Val AUC: 0.9214
✅ Best model saved (AUC improved)
----------------------------------------
Epoch 4/30
Train Loss: 0.2418
Val Accuracy: 0.6280
Val F1: 0.7238
Val AUC: 0.9230
✅ Best model saved (AUC improved)
----------------------------------------
Epoch 5/30
Train Loss: 0.2395
Val Accuracy: 0.6294
Val F1: 0.7174
Val AUC: 0.9230
⚠️ No AUC improvement (1/5)
----------------------------------------
Epoch 6/30
Train Loss: 0.2391
Val Accuracy: 0.6257
Val F1: 0.7118
Val AUC: 0.9231
✅ Best model saved (AUC improved)
------------------

In [25]:
import numpy as np

# 1. Convert probabilities to binary predictions
preds = (all_probs > 0.5).astype(int)

# 2. Identify which samples are single-label vs multicategorical
num_labels_per_patient = all_labels.sum(axis=1)
multi_mask = num_labels_per_patient > 1
single_mask = num_labels_per_patient == 1

# 3. Calculate metrics for Multicategorical Cases
y_true_multi = all_labels[multi_mask]
y_pred_multi = preds[multi_mask]

# How many labels did the model find out of the total possible?
# (Intersection over True Labels)
recall_per_sample = (y_true_multi * y_pred_multi).sum(axis=1) / y_true_multi.sum(axis=1)
avg_recall_multi = recall_per_sample.mean()

# Exact Match: Model got all labels right for that patient
exact_match_multi = (y_true_multi == y_pred_multi).all(axis=1).mean()

# Partial Match: Model got at least one label right
partial_match_multi = ((y_true_multi == 1) & (y_pred_multi == 1)).any(axis=1).mean()

# 4. Print the "Evidence"
print("--- ANALYSIS OF MULTICATEGORICAL CASES ---")
print(f"Number of multicategorical patients in test set: {multi_mask.sum()}")
print(f"Exact Match Rate (Perfectly caught all labels): {exact_match_multi:.2%}")
print(f"Partial Match Rate (Caught at least one label): {partial_match_multi:.2%}")
print(f"Average % of labels caught per patient: {avg_recall_multi:.2%}")

# 5. Show specific failures (e.g., predicted MI but missed CD)
print("\nExample of partial successes (True vs Predicted):")
indices = np.where(multi_mask)[0][:5] # Look at the first 5 multi-label patients
for idx in indices:
    t = [classes[i] for i, val in enumerate(all_labels[idx]) if val == 1]
    p = [classes[i] for i, val in enumerate(preds[idx]) if val == 1]
    print(f"True: {t} | Pred: {p}")

--- ANALYSIS OF MULTICATEGORICAL CASES ---
Number of multicategorical patients in test set: 508
Exact Match Rate (Perfectly caught all labels): 27.56%
Partial Match Rate (Caught at least one label): 91.14%
Average % of labels caught per patient: 62.78%

Example of partial successes (True vs Predicted):
True: ['CD', 'HYP', 'MI', 'STTC'] | Pred: ['CD', 'MI', 'STTC']
True: ['CD', 'NORM'] | Pred: ['NORM']
True: ['HYP', 'MI', 'STTC'] | Pred: ['HYP', 'MI', 'STTC']
True: ['HYP', 'MI', 'STTC'] | Pred: ['HYP', 'MI', 'STTC']
True: ['HYP', 'STTC'] | Pred: ['HYP', 'STTC']


In [26]:
# Comparison table (xresnet1d101 vs fusion model)
# -------------------------
# ECG-only model metrics
# -------------------------

# Create model exactly like during training
ecg_model = xresnet1d101(input_channels=12, num_classes=5).to(DEVICE)

# Load weights, ignore mismatches if you modified later
checkpoint = torch.load("../outputs/best_ecg_model.pt", map_location=DEVICE)
ecg_model.load_state_dict(checkpoint, strict=False)

ecg_model.eval()

from torch.utils.data import DataLoader
from utils.data_loader import PTBXL_Dataset

# ECG-only dataset
test_dataset = PTBXL_Dataset(X_test, y_test)
test_dl = DataLoader(test_dataset, batch_size=32)

# Now run predictions
all_ecg_probs, all_ecg_labels = get_predictions(ecg_model, test_dl, device=DEVICE)

print("\n=== ECG MODEL RESULTS ===")
ecg_model.load_state_dict(torch.load("../outputs/best_ecg_model.pt", map_location=DEVICE))
ecg_model.eval()

all_ecg_probs, all_ecg_labels = get_predictions(ecg_model, test_dl, device=DEVICE)
ecg_metrics = compute_all_metrics(all_ecg_labels, all_ecg_probs)

for k, v in ecg_metrics.items():
    print(f"{k}: {v:.4f}")


# -------------------------
# Comparison table
# -------------------------
print("\n=== MODEL COMPARISON ===")
for key in ecg_metrics.keys():
    print(f"{key:20s} | ECG: {ecg_metrics[key]:.4f} | Fusion: {fusion_metrics[key]:.4f}")

C:\Users\inaki\AppData\Local\Temp\ipykernel_6540\1251860348.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../outputs/best_ecg_model.pt", map_


=== ECG MODEL RESULTS ===


C:\Users\inaki\AppData\Local\Temp\ipykernel_6540\1251860348.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ecg_model.load_state_dict(torch.load("../outputs/best_ecg_mo

Accuracy: 0.6260
F1: 0.7251
Precision (PPV): 0.7918
Sensitivity (Recall): 0.6862
Specificity: 0.9265
MCC: 0.6938
AUC: 0.9224

=== MODEL COMPARISON ===
Accuracy             | ECG: 0.6260 | Fusion: 0.6256
F1                   | ECG: 0.7251 | Fusion: 0.7197
Precision (PPV)      | ECG: 0.7918 | Fusion: 0.8042
Sensitivity (Recall) | ECG: 0.6862 | Fusion: 0.6689
Specificity          | ECG: 0.9265 | Fusion: 0.9328
MCC                  | ECG: 0.6938 | Fusion: 0.6910
AUC                  | ECG: 0.9224 | Fusion: 0.9227


The fusion model achieves comparable AUC performance to the ECG-only model, while improving sensitivity. This indicates that incorporating metadata helps detect more positive cases (increases sensitivity), at the cost of a slight increase in false positives.

### Statistical Comparison (Bootstrap T-Test)

In [29]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def bootstrap_comparison(all_labels, ecg_probs, fusion_probs, n_iterations=10000, alpha=0.05):
    # Store bootstrap differences (Fusion - ECG)
    diffs = {
        'Accuracy': [],
        'F1': [],
        'AUC': []
    }

    # Convert probabilities to binary predictions once (threshold = 0.5)
    ecg_preds = (ecg_probs > 0.5).astype(int)
    fusion_preds = (fusion_probs > 0.5).astype(int)

    print(f"Running {n_iterations} bootstrap iterations...\n")

    for _ in range(n_iterations):
        # Resample indices with replacement
        indices = resample(np.arange(len(all_labels)))

        # Get resampled ground truth
        y_true = all_labels[indices]

        # ---- ECG model metrics ----
        acc_ecg = accuracy_score(y_true, ecg_preds[indices])
        f1_ecg = f1_score(y_true, ecg_preds[indices], average='macro')
        auc_ecg = roc_auc_score(y_true, ecg_probs[indices], average='macro')

        # ---- Fusion model metrics ----
        acc_fusion = accuracy_score(y_true, fusion_preds[indices])
        f1_fusion = f1_score(y_true, fusion_preds[indices], average='macro')
        auc_fusion = roc_auc_score(y_true, fusion_probs[indices], average='macro')

        # Store metric differences (Fusion - ECG)
        diffs['Accuracy'].append(acc_fusion - acc_ecg)
        diffs['F1'].append(f1_fusion - f1_ecg)
        diffs['AUC'].append(auc_fusion - auc_ecg)

    print("=== BOOTSTRAP RESULTS (Fusion - ECG) ===")
    print(f"{'Metric':12} | {'Mean Diff':10} | {'CI Lower':10} | {'CI Upper':10} | {'Significant?'}")
    print("-" * 75)

    for metric in diffs:
        values = np.array(diffs[metric])

        # Mean difference
        mean_diff = np.mean(values)

        # Confidence interval (percentile method)
        lower = np.percentile(values, 100 * (alpha / 2))
        upper = np.percentile(values, 100 * (1 - alpha / 2))

        # Check if CI includes zero
        significant = "YES" if not (lower <= 0 <= upper) else "NO"

        print(f"{metric:12} | {mean_diff:.6f} | {lower:.6f} | {upper:.6f} | {significant}")


# =========================
# RUN
# =========================
# all_ecg_labels → ground truth labels
# all_ecg_probs  → predicted probabilities from ECG model
# all_probs      → predicted probabilities from Fusion model

bootstrap_comparison(all_ecg_labels, all_ecg_probs, all_probs)

Running 10000 bootstrap iterations...

=== BOOTSTRAP RESULTS (Fusion - ECG) ===
Metric       | Mean Diff  | CI Lower   | CI Upper   | Significant?
---------------------------------------------------------------------------
Accuracy     | -0.000406 | -0.011374 | 0.010464 | NO
F1           | -0.005341 | -0.013558 | 0.002747 | NO
AUC          | 0.000321 | -0.000924 | 0.001559 | NO


Despite incorporating multimodal information, the fusion model did not yield statistically significant improvements over the ECG-only model, suggesting that the additional modality may not provide complementary predictive value in this setting.

## 9. Save best model

In [28]:
best_auc = 0
patience = 5
epochs_no_improve = 0

for epoch in range(num_epochs):
    train_loss = train_one_epoch(fusion_model, train_loader)
    val_acc, val_f1, val_auc = evaluate(fusion_model, val_loader)

    # Save best model based on AUC
    if val_auc > best_auc:
        best_auc = val_auc
        epochs_no_improve = 0
        torch.save(fusion_model.state_dict(), "../outputs/best_fusion_model.pt")
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1}: F1={val_f1:.4f}, AUC={val_auc:.4f}")

    # Early stopping
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered at epoch {epoch+1} (no AUC improvement).")
        break

Epoch 1: F1=0.7234, AUC=0.9243
Epoch 2: F1=0.7222, AUC=0.9237
Epoch 3: F1=0.7122, AUC=0.9233
Epoch 4: F1=0.7142, AUC=0.9236
Epoch 5: F1=0.7163, AUC=0.9235
Epoch 6: F1=0.7072, AUC=0.9236
Early stopping triggered at epoch 6 (no AUC improvement).


The fusion model achieves comparable AUC performance to the ECG-only model, while improving sensitivity. This indicates that incorporating metadata helps detect more positive cases (increases sensitivity), at the cost of a slight increase in false positives.